In [ ]:
import albumentations as A
import matplotlib.pyplot as plt
import os
from skimage import io
import numpy as np
from numpy import ndarray
from skimage.measure import label
from skimage.segmentation import expand_labels
from skimage.morphology import remove_small_objects
from skimage.segmentation import watershed
from skimage.filters import sobel

from image_processing import normalize, fill_labels
from skeleton_aware_distance_transform import LabelDistanceTransforms

In [ ]:
TRAIN_IMG_DIR = "data/test_images/"
TRAIN_SEM_DIR = "data/test_masks/"
TRAIN_MASK_DIR = "data/test_labels/"
index = 0
fill_gt = False

image_path = os.path.join(TRAIN_IMG_DIR, os.listdir(TRAIN_IMG_DIR)[index])
mask_path = os.path.join(TRAIN_SEM_DIR, os.listdir(TRAIN_SEM_DIR)[index])
mask2_path = os.path.join(TRAIN_MASK_DIR, os.listdir(TRAIN_MASK_DIR)[index])

image = normalize(io.imread(image_path).astype(np.float32))
mask = io.imread(mask_path).astype(np.float32)
mask2 = io.imread(mask2_path).astype(np.int8)

print(os.listdir(TRAIN_IMG_DIR)[index])

In [ ]:
if fill_gt:
    mask2 = fill_labels(mask2)

In [ ]:
final_dist_trans, bdt, sdt = LabelDistanceTransforms(mask2[:-1,:-1], 0.2, True, False, True).skeleton_aware_dist_trans()

In [ ]:
# plot example image, semantic mask and overlay
fig, ax = plt.subplots(1,4, figsize=(16, 4))
fig.suptitle("Skeleton-aware distance transform")
ax[0].imshow(image, cmap='gray')
ax[0].set_title("Image")
ax[1].imshow(bdt, cmap='viridis')
ax[1].set_title("Boundary distance transform")
ax[2].imshow(sdt, "viridis")
ax[2].set_title("Skeleton distance transform")
ax[3].imshow(final_dist_trans, cmap='viridis')
ax[3].set_title("Final distance transform")
plt.tight_layout
plt.show()

In [ ]:
type(final_dist_trans)

In [ ]:
# plot example image, semantic mask and overlay
fig, ax = plt.subplots(1,4, figsize=(16, 4))
fig.suptitle("Skeleton-aware distance transform")
ax[0].imshow(image, cmap='gray')
ax[0].set_title("Image")
ax[1].imshow(fill_labels(mask2[:-1,:-1].astype(np.int16)), cmap='plasma')
ax[1].set_title("Ground truth")
ax[2].imshow(bdt, cmap='viridis')
ax[2].set_title("Boundary distance transform")
ax[3].imshow(sdt, "viridis")
ax[3].set_title("Skeleton distance transform")
plt.tight_layout
plt.show()

In [ ]:
print(final_dist_trans.min())
print(final_dist_trans.max())

In [ ]:
# Export intermediate results

from PIL import Image

# semantic, distance transform and instance predictions
image_bdt = Image.fromarray(bdt.astype(np.float32), 'F') # 32-bit
image_bdt.save(r'figures\bdt.tif')
image_sdt = Image.fromarray(sdt.astype(np.float32), 'F') # 32-bit
image_sdt.save(r'figures\sdt.tif')
image_final_dist_trans = Image.fromarray(final_dist_trans.astype(np.float32), 'F') # 32-bit
image_final_dist_trans.save(r'figures\final_sadt.tif')
image_pred_instance = Image.fromarray(mask2.astype(np.int8), 'L') # 16-bit
image_pred_instance.save(r'figures\ground_truth_instance.tif')

In [ ]:
transform = A.Compose([
    A.RandomCrop(
        height=final_dist_trans.shape[0],
        width=final_dist_trans.shape[1],
        always_apply=True),  
    ], p=1)

transformed = transform(image=image, masks=[mask, mask2, final_dist_trans])

image_transformed = transformed['image']
mask_transformed = transformed['masks'][0]
mask2_transformed = transformed['masks'][1]
mask3_transformed = transformed['masks'][2]

In [ ]:
# plot example image, semantic mask and overlay
fig, ax = plt.subplots(2,2, figsize=(8, 8))
fig.suptitle("Multimask example")
ax[0,0].imshow(image_transformed, cmap='gray')
ax[0,0].set_title("Image")
ax[0,1].imshow(mask_transformed, "viridis")
ax[0,1].set_title("Semantic Mask")
ax[1,0].imshow(mask2_transformed, cmap='magma')
ax[1,0].set_title("Instance Mask")
ax[1,1].imshow(mask3_transformed, cmap='plasma')
ax[1,1].set_title("Distance transform")
plt.tight_layout
plt.show()

In [ ]:
plt.imshow(final_dist_trans, cmap='plasma')
plt.colorbar()
print(np.unique(final_dist_trans))

In [ ]:
def postprocessing_sdt(distancemap):
    mask_fibre = distancemap > 0.05
    mask_inreg = np.logical_and(distancemap >= 0.35, mask_fibre)
    mask_inreg = label(mask_inreg)
    mask_inreg = remove_small_objects(mask_inreg, min_size=50, connectivity=1)
    mask_inreg = label(mask_inreg)
    labels = watershed(distancemap, mask_inreg, mask=mask_fibre, compactness=10)
    filled_labels = fill_labels(labels)
    expanded_labels = expand_labels(label_image=filled_labels, distance=1)
    return filled_labels

result = postprocessing_sdt(mask3_transformed)
plt.imshow(result)